# SOLA Method with Noise on a 3D Box Domain

## The Continuous Problem

We address the task of estimating a finite-dimensional **property** of an unknown spatial model from **noisy observations**. The model space is the $L^2$ space on a unit cube.

Let $m \in \mathcal{M} = L^2([0,1]^3)$ be the unknown model, $\tilde{\mathbf{d}} \in \mathcal{D} \cong \mathbb{R}^{N_d}$ be the observed data, and $\tilde{\mathbf{p}} \in \mathcal{P} \cong \mathbb{R}^{N_p}$ be the desired properties.

### Forward Operator

The forward map is

$$G : \mathcal{M} \to \mathcal{D}, \qquad [G(m)]_i = \int_{[0,1]^3} K_i(\mathbf{x})\, m(\mathbf{x})\, \mathrm{d}V$$

where $K_i$ are **Gaussian sensitivity kernels**. Noisy observations:

$$\tilde{\mathbf{d}} = G(\bar{m}) + \boldsymbol{\eta}, \qquad \boldsymbol{\eta} \sim \mathcal{N}(\mathbf{0}, \mathbf{C}_\mathcal{D})$$

### SOLA Approach

We do not reconstruct $\bar{m}$ directly. Instead we seek a surrogate operator $\mathbf{X}: \mathcal{D} \to \mathcal{P}$ such that $\mathcal{T} \approx \mathbf{X} G$, minimising

$$\min_{\mathbf{X}} \left[\|\mathcal{T} - \mathbf{X}G\|_{HS}^2 + \mathrm{Tr}(\mathbf{X}\mathbf{C}_\mathcal{D}\mathbf{X}^*)\right]$$

The **analytical solution** is

$$\boxed{\mathbf{X} = \mathcal{T}\, G^*\!\left(G G^* + \mathbf{C}_\mathcal{D}\right)^{-1}}$$

The property estimate is obtained by pushing the data measure forward:

$$\mu_\mathcal{P} = \mathbf{X}_* \mu_\mathcal{D} = \mathcal{N}(\mathbf{X}\tilde{\mathbf{d}},\; \mathbf{X}\mathbf{C}_\mathcal{D}\mathbf{X}^*)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

from pygeoinf import (
    EuclideanSpace,
    LinearOperator,
    CholeskySolver,
    GaussianMeasure,
)
from pygeoinf3d import (
    BoxDomain,
    Lebesgue3D,
    Function3D,
    FunctionToKernelAdapter,
    SOLAOperator3D,
)
from pygeoinf3d.providers.function_providers import GaussianProvider3D
from pygeoinf3d.plot import (
    plot_function3d_axis_slider,
    compare_function3d_axis_sliders,
    plot_function3d_axis_dropdown_slider,
)

np.random.seed(42)

N_SLICES = 10     # slider resolution — keep low for fast rendering

## Spaces and Operators

We work on the unit cube $[0,1]^3$ with an $L^2$ model space.

* **$G$** — forward operator: $N_d$ Gaussian sensitivity kernels at random locations
* **$\mathcal{T}$** — target/property operator: $N_p$ tighter Gaussians on a $2\times2\times2$ regular grid

In [ ]:
# ── Domain & model space ─────────────────────────────────────────────────────
box   = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# ── Data space ────────────────────────────────────────────────────────────────
N_d = 35
D = EuclideanSpace(N_d)

# ── Property space ────────────────────────────────────────────────────────────
# 2×2×2 grid of target locations
target_grid = np.array([[x, y, z]
                         for x in [0.25, 0.75]
                         for y in [0.25, 0.75]
                         for z in [0.25, 0.75]])
N_p = len(target_grid)  # 8
P = EuclideanSpace(N_p)

# ── Sensitivity kernels for G ─────────────────────────────────────────────────
centers_G = np.random.uniform(0.1, 0.9, size=(N_d, 3))
sigmas_G  = np.random.uniform(0.12, 0.20, size=(N_d, 3))
fp_G   = GaussianProvider3D(box, centers=centers_G, sigmas=sigmas_G, normalized=True)
kp_G   = FunctionToKernelAdapter(fp_G, space)
G      = SOLAOperator3D(space, D, kp_G)

# ── Target kernels for T ──────────────────────────────────────────────────────
fp_T   = GaussianProvider3D(box, centers=target_grid, sigmas=0.08, normalized=True)
kp_T   = FunctionToKernelAdapter(fp_T, space)
T_op   = SOLAOperator3D(space, P, kp_T)

print(f"Forward operator G  : {type(G.domain).__name__} → R^{N_d}")
print(f"Target operator T   : {type(T_op.domain).__name__} → R^{N_p}")
print(f"Target locations    : {N_p} (2×2×2 grid)")

## Sensitivity and Target Kernel Visualisation

We visualise selected kernels as mid-plane ($z = 0.5$) slices.

In [ ]:
# ── All 35 sensitivity kernels — dropdown + z-slider ─────────────────────────
g_kernels = [kp_G.get_kernel(i).kernel for i in range(N_d)]
g_labels  = [f"G kernel {i}  ({centers_G[i,0]:.2f},{centers_G[i,1]:.2f},{centers_G[i,2]:.2f})"
             for i in range(N_d)]
_ = plot_function3d_axis_dropdown_slider(
    g_kernels, axis='z', n_slices=N_SLICES, resolution=80,
    labels=g_labels, title="Sensitivity kernels  (select from dropdown)",
    show_plot=True,
)

# ── All 8 target kernels — dropdown + z-slider ────────────────────────────────
t_kernels = [kp_T.get_kernel(k).kernel for k in range(N_p)]
t_labels  = [f"T k={k} ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})"
             for k in range(N_p)]
_ = plot_function3d_axis_dropdown_slider(
    t_kernels, axis='z', n_slices=N_SLICES, resolution=80,
    labels=t_labels, title="Target kernels  (select from dropdown)",
    show_plot=True,
)


## Synthetic Data

We define a multi-blob true model $m_{\text{true}}$ and generate noisy synthetic data
$$
d = G\,m_{\text{true}} + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, C_D).
$$

In [ ]:
def gaussian_blob(x, center, sigma):
    """Evaluate a normalised 3-D Gaussian at points x (shape n×3)."""
    diff = x - center[None, :]
    return np.exp(-0.5 * np.sum((diff / sigma) ** 2, axis=1))

def true_model_fn(x):
    """Two positive + one negative Gaussian blob."""
    if np.ndim(x) == 1:
        x = x[None, :]
    vals = (
        + 1.5 * gaussian_blob(x, np.array([0.30, 0.30, 0.50]), 0.12)
        + 1.0 * gaussian_blob(x, np.array([0.70, 0.65, 0.55]), 0.10)
        - 0.8 * gaussian_blob(x, np.array([0.55, 0.40, 0.45]), 0.08)
    )
    return vals.squeeze()

m_true = Function3D(box, evaluate_callable=true_model_fn)

# Clean and noisy data
d_bar     = G(m_true)
noise_std = 0.05 * np.abs(d_bar).max()
d_tilde   = d_bar + np.random.normal(0.0, noise_std, size=N_d)

print(f"Noise std : {noise_std:.4f}")
print(f"SNR       : {np.linalg.norm(d_bar) / np.linalg.norm(d_tilde - d_bar):.1f}")

# ── Interactive z-slider for true model ───────────────────────────────────────
plot_function3d_axis_slider(
    m_true, axis='z', n_slices=N_SLICES, resolution=80,
    title="True model  (z-axis slider)", show_plot=False,
)

## Data Noise Model

We model the data noise as a Gaussian measure on the data space $D = \mathbb{R}^{N_d}$:
$$
\mu_D = \mathcal{N}(\tilde{d},\, C_D), \qquad C_D = \sigma_n^2 I.
$$
This encodes our knowledge about both the noise level and the observed data.

In [ ]:
C_D_matrix = noise_std**2 * np.eye(N_d)
gaussian_D  = GaussianMeasure.from_covariance_matrix(D, C_D_matrix,
                                                      expectation=d_tilde)

print(f"Data measure expectation (first 5): {gaussian_D.expectation[:5].round(4)}")
print(f"Data covariance diagonal (var):     {C_D_matrix[0, 0]:.6f}")

## SOLA Operator Construction

The SOLA solution operator is
$$
X = T\,G^*(G G^* + C_D)^{-1}.
$$

We build it step by step:
1. **Gram operator** $\Lambda = G G^*$ (data-space self-adjoint operator).
2. **Regularised inverse** $W^{-1} = (\Lambda + C_D)^{-1}$ via Cholesky.
3. **Generalised inverse** $G^{-1} = G^*\,W^{-1}$.
4. **SOLA operator** $X = T\,G^{-1}$.

In [ ]:
cholesky_solver = CholeskySolver(galerkin=True, parallel=True, n_jobs=-1)

# Step 1: Gram operator  Λ : D → D
Lambda = G @ G.adjoint

# Step 2: W_inv = (Λ + C_D)^{-1}  :  D → D
W_inv = cholesky_solver(Lambda + gaussian_D.covariance)

# Step 3: generalised inverse  G_inv = G* W_inv  :  D → M
G_inv = G.adjoint @ W_inv

# Step 4: SOLA operator  X = T G_inv  :  D → P
X = T_op @ G_inv

print("SOLA operator X assembled.")
print(f"  domain   : R^{N_d}  (data space)")
print(f"  codomain : R^{N_p}  (property space)")

## Model Reconstruction (Optional)

The generalised inverse $G^{-1}$ also provides an approximate model reconstruction
$\hat{m} = G^{-1}(\tilde{d})$.
This is a Function3D living in the model space.

In [ ]:
m_hat = G_inv(d_tilde)   # Function3D

# Data residual
residual = np.linalg.norm(G(m_hat) - d_tilde) / np.linalg.norm(d_tilde)
print(f"Relative data residual: {residual:.4f}")

# ── Side-by-side interactive slider: true model vs. reconstruction ───────────
compare_function3d_axis_sliders(
    [m_true, m_hat], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["True model", "Reconstruction"],
    show_plot=False,
    width=1000, height=500,
)

## Property Inference with Uncertainty

The SOLA estimator $\hat{p} = X(\tilde{d})$ is a linear function of the data.
Because the data follows a Gaussian measure $\mu_D$, the inferred properties also follow a Gaussian:
$$
\mu_P = X_* \mu_D = \mathcal{N}(X\tilde{d},\;\; X C_D X^*).
$$

The posterior standard deviations quantify the propagated data noise.

In [ ]:
# Push the data measure forward through X
property_measure = gaussian_D.affine_mapping(operator=X)

p_mean    = property_measure.expectation                    # shape (N_p,)
# Assemble covariance matrix by applying the operator to each basis vector
P_cov_mat = property_measure.covariance.matrix(dense=True, galerkin=False, parallel=True, n_jobs=-1)  # (N_p, N_p)
p_std     = np.sqrt(np.diag(P_cov_mat))

# True property values  p_true = T(m_true)
p_true = T_op(m_true)

print("Property  |  true    |  mean    |  std")
print("-" * 48)
for k in range(N_p):
    cx, cy, cz = target_grid[k]
    print(f"  k={k}  ({cx},{cy},{cz})  |  {p_true[k]:.4f}  |  {p_mean[k]:.4f}  |  {p_std[k]:.4f}")

# ── Helper: wrap N_p property values as a Function3D on the 2×2×2 grid ─────
_grid_ax = np.array([0.25, 0.75])   # x/y/z coordinates of the target grid

def _make_prop_fn(values):
    """Trilinearly interpolate property values on the 2×2×2 target grid."""
    grid_vals = values.reshape(2, 2, 2)   # axis order: x, y, z
    _interp = RegularGridInterpolator(
        (_grid_ax, _grid_ax, _grid_ax), grid_vals,
        method='linear', bounds_error=False, fill_value=0.0,
    )
    def _fn(x):
        pts = np.clip(np.atleast_2d(x), 0.25, 0.75)   # nearest-edge extrapolation
        return _interp(pts).squeeze()
    return Function3D(box, evaluate_callable=_fn)

p_mean_field = _make_prop_fn(p_mean)
p_2std_field = _make_prop_fn(2 * p_std)

# ── 3D slider: SOLA estimate and uncertainty side-by-side ────────────────────
compare_function3d_axis_sliders(
    [p_mean_field, p_2std_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["SOLA estimate (mean)", "Uncertainty (2σ)"],
    show_plot=False,
    width=1000, height=500,
)


## Unimodularity Constraint

A SOLA estimator is **unimodular** if $XG \mathbf{1} = \mathbf{1}$, i.e., a uniform model maps to unit
property values.  The unconstrained $X$ generally violates this.  We apply a rank-1 correction:
$$
X_{\text{uni}} = X - \frac{(X\,G\mathbf{1} - \mathbf{1})\,u^T}{\langle G\mathbf{1},\, u\rangle}, \quad
u = W^{-1}(G\mathbf{1}).
$$

In [ ]:
ones = Function3D(box, evaluate_callable=lambda x: np.ones(np.atleast_2d(x).shape[0]))

v    = G(ones)                          # G applied to constant field → R^{N_d}
w    = np.ones(N_p)                     # desired unimodular output
u    = W_inv(v)                         # W^{-1} v → R^{N_d}
beta = float(np.dot(v, u))             # ⟨v, u⟩ scalar

# Rank-1 correction operator  D → P
correction_matrix = np.outer((X(v) - w) / beta, u)   # shape (N_p, N_d)
correction    = LinearOperator.from_matrix(D, P, correction_matrix)
X_unimodular  = X - correction

# Verify unimodularity
check = X_unimodular(v)
print(f"X_uni(G 1) — should equal ones: {check.round(4)}")

# Push-forward with unimodular X
property_measure_uni = gaussian_D.affine_mapping(operator=X_unimodular)
p_mean_uni = property_measure_uni.expectation
P_cov_uni_mat = property_measure_uni.covariance.matrix(dense=True, galerkin=False, parallel=True, n_jobs=-1)
p_std_uni  = np.sqrt(np.diag(P_cov_uni_mat))

p_mean_uni_field = _make_prop_fn(p_mean_uni)
p_2std_uni_field = _make_prop_fn(2 * p_std_uni)

# ── Mean: unconstrained vs unimodular ─────────────────────────────────────────
compare_function3d_axis_sliders(
    [p_mean_field, p_mean_uni_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["Mean — unconstrained X", "Mean — unimodular X"],
    show_plot=False,
    width=1000, height=500,
)

# ── Uncertainty: unconstrained vs unimodular ──────────────────────────────────
compare_function3d_axis_sliders(
    [p_2std_field, p_2std_uni_field], axis='z', n_slices=N_SLICES, resolution=80,
    titles=["2σ — unconstrained X", "2σ — unimodular X"],
    show_plot=False,
    width=1000, height=500,
)


## Resolving Kernels

The **resolving kernel** for property $k$ is the function
$$
A_k(x) = \bigl(X G\bigr)^*(e_k)(x) = \bigl(G^*\,X^*\,e_k\bigr)(x),
$$
where $e_k$ is the $k$-th standard basis vector in property space $P$.
Each $A_k$ is a weighted superposition of the sensitivity kernels; ideally it closely
approximates the corresponding target kernel $T_k$.

In [ ]:
# (X G)^* = G^* X^*   maps  P → M
A_adjoint = (X @ G).adjoint

# Collect ALL resolving and target kernels
rk_fns  = [A_adjoint(np.eye(N_p)[k]) for k in range(N_p)]
tgt_fns = [kp_T.get_kernel(k).kernel  for k in range(N_p)]

# ── Resolving + target kernels in a single dropdown+slider ───────────────────
# Interleave: Resolving k=0, Target k=0, Resolving k=1, Target k=1, ...
combined_fns    = [fn for pair in zip(rk_fns, tgt_fns) for fn in pair]
combined_labels = [
    label
    for k in range(N_p)
    for label in [
        f"Resolving k={k} ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})",
        f"Target    k={k} ({target_grid[k,0]:.2f},{target_grid[k,1]:.2f},{target_grid[k,2]:.2f})",
    ]
]

_ = plot_function3d_axis_dropdown_slider(
    combined_fns, axis='z', n_slices=N_SLICES, resolution=80,
    labels=combined_labels,
    title="Resolving & target kernels  (select from dropdown)",
    per_function_colorscale=True,
    show_plot=False,
)

print("Ideal: each resolving kernel should be concentrated near its target centre.")
